# Checkpoint 7: Churn / Retention Modeling & Baseline Benchmarking

**Project**: AI-Powered E-commerce Customer Segmentation and Churn Analysis  
**Dataset**: Brazilian E-Commerce Public Dataset by Olist  
**Scope**: Dual-window evaluation ($W=120\text{d}$ and $W=180\text{d}$) of predictive churn/return classification against 3 pre-registered baselines and 3 candidate ML models.

### Methodology & Leakage Safeguards
- **Entity**: Customer-level evaluation using `customer_unique_id`.
- **Target**: Binary indicator `return_within_window` $\in \{0, 1\}$, where $1$ indicates at least one delivered purchase in $(T_{obs}, T_{obs} + W]$ and $0$ indicates an observed non-return / churn-risk proxy.
- **Zero Temporal Leakage**: Features use strictly transactions with `order_purchase_timestamp` $\le T_{obs}$; target uses strictly $T_{obs} < \text{timestamp} \le T_{obs} + W$.
- **Zero Right-Censoring**: $T_{obs} = T_{max} - W$ guarantees complete follow-up for all eligible customers. Post-cutoff entrants are excluded.
- **Evaluation Integrity**: Evaluated against Majority-Class, Stratified Dummy, and Frozen RFM Heuristic (`recency_days <= 90.0`). Thresholds tuned strictly on training folds; test set evaluated once.
- **Privacy & Immutability**: All 9 raw CSV hashes verified immutable; zero individual PII or customer identifiers persisted.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Load pre-computed aggregate checkpoint
checkpoint_path = Path('../data/checkpoints/07_churn_modeling.json')
with open(checkpoint_path, 'r', encoding='utf-8') as f:
    cp7 = json.load(f)

print(f"Checkpoint: {cp7['checkpoint']}")
print(f"Description: {cp7['checkpoint_description']}")
print(f"Audit Timestamp (UTC): {cp7['audit_timestamp_utc']}")
print(f"Raw Data Immutability Verified: {cp7['raw_data_immutability_verified']}")

## 1. Dual-Window Temporal Architecture & Leakage Verification

Verification that all pre-cutoff features terminate $\le T_{obs}$ and all target transactions occur strictly $> T_{obs}$ without right-censoring.

In [ ]:
t120 = cp7['candidate_window_120d']['temporal_leakage_audit']
t180 = cp7['candidate_window_180d']['temporal_leakage_audit']

leakage_summary = [
    {'Window Dimension': 'Candidate Window Days (W)', '120-Day Window': '120 days', '180-Day Window': '180 days'},
    {'Window Dimension': 'Observation Cutoff (T_obs)', '120-Day Window': t120['cutoff_timestamp_T_obs'], '180-Day Window': t180['cutoff_timestamp_T_obs']},
    {'Window Dimension': 'Max Feature Timestamp', '120-Day Window': t120['max_feature_order_timestamp'], '180-Day Window': t180['max_feature_order_timestamp']},
    {'Window Dimension': 'Min Target Timestamp', '120-Day Window': t120['min_target_order_timestamp'], '180-Day Window': t180['min_target_order_timestamp']},
    {'Window Dimension': 'Max Target Timestamp', '120-Day Window': t120['max_target_order_timestamp'], '180-Day Window': t180['max_target_order_timestamp']},
    {'Window Dimension': 'Temporal Assertion Passed', '120-Day Window': str(t120['temporal_assertion_passed']), '180-Day Window': str(t180['temporal_assertion_passed'])},
    {'Window Dimension': 'Eligible Customers', '120-Day Window': f"{t120['eligible_customer_count']:,}", '180-Day Window': f"{t180['eligible_customer_count']:,}"},
    {'Window Dimension': 'Excluded Post-Cutoff Entrants', '120-Day Window': f"{t120['post_cutoff_entrants_excluded']:,}", '180-Day Window': f"{t180['post_cutoff_entrants_excluded']:,}"},
    {'Window Dimension': 'Right-Censored Customers', '120-Day Window': str(t120['right_censored_eligible_customers']), '180-Day Window': str(t180['right_censored_eligible_customers'])},
    {'Window Dimension': 'Return Cases (y=1)', '120-Day Window': f"{t120['return_cases_count']:,}", '180-Day Window': f"{t180['return_cases_count']:,}"},
    {'Window Dimension': 'Return Incidence (%)', '120-Day Window': f"{t120['return_incidence_pct']:.4f}%", '180-Day Window': f"{t180['return_incidence_pct']:.4f}%"}
]

df_leakage = pd.DataFrame(leakage_summary)
df_leakage

## 2. Candidate Window 120-Day Benchmarking ($W=120\text{d}$)

Performance comparison across 3 baselines and 3 candidate ML models on held-out 20% stratified test set ($N_{\text{test}}=13,796$ customers, 108 positive returners).

In [ ]:
m120 = cp7['candidate_window_120d']['modeling_experiment']['models_evaluated']

rows_120 = []
for m_key, m_val in m120.items():
    tm = m_val['test_metrics']
    rows_120.append({
        'Model / Baseline': m_val['model_name'],
        'Type': m_val['type'].title(),
        'PR-AUC': f"{tm['pr_auc']:.4f}",
        'ROC-AUC': f"{tm['roc_auc']:.4f}",
        'Top-Decile Lift': f"{tm['top_decile_lift']:.2f}x",
        'Top-Decile Gain (%)': f"{tm['top_decile_cumulative_gain']*100:.1f}%",
        'Precision': f"{tm['precision']:.4f}",
        'Recall': f"{tm['recall']:.4f}",
        'F1-Score': f"{tm['f1_score']:.4f}",
        'Brier Score': f"{tm['brier_score']:.5f}",
        'Accuracy': f"{tm['accuracy_descriptive']*100:.2f}%",
        'Train-Test Gap (PR-AUC)': f"{m_val['train_vs_test_pr_auc_divergence']:.4f}"
    })

df_m120 = pd.DataFrame(rows_120)
df_m120

## 3. Candidate Window 180-Day Benchmarking ($W=180\text{d}$)

Performance comparison across 3 baselines and 3 candidate ML models on held-out 20% stratified test set ($N_{\text{test}}=11,182$ customers, 131 positive returners).

In [ ]:
m180 = cp7['candidate_window_180d']['modeling_experiment']['models_evaluated']

rows_180 = []
for m_key, m_val in m180.items():
    tm = m_val['test_metrics']
    rows_180.append({
        'Model / Baseline': m_val['model_name'],
        'Type': m_val['type'].title(),
        'PR-AUC': f"{tm['pr_auc']:.4f}",
        'ROC-AUC': f"{tm['roc_auc']:.4f}",
        'Top-Decile Lift': f"{tm['top_decile_lift']:.2f}x",
        'Top-Decile Gain (%)': f"{tm['top_decile_cumulative_gain']*100:.1f}%",
        'Precision': f"{tm['precision']:.4f}",
        'Recall': f"{tm['recall']:.4f}",
        'F1-Score': f"{tm['f1_score']:.4f}",
        'Brier Score': f"{tm['brier_score']:.5f}",
        'Accuracy': f"{tm['accuracy_descriptive']*100:.2f}%",
        'Train-Test Gap (PR-AUC)': f"{m_val['train_vs_test_pr_auc_divergence']:.4f}"
    })

df_m180 = pd.DataFrame(rows_180)
df_m180

## 4. Side-by-Side Candidate Window Comparison (120d vs 180d)

Comprehensive comparison of the two candidate prediction horizons across data coverage, class incidence, and model behavior.

In [ ]:
w_comp = cp7['methodological_decisions']['candidate_window_comparison_summary']

w_rows = [
    {'Metric': 'Eligible Population Coverage', '120-Day Window': f"{w_comp['eligible_customers']['120d']:,} (73.9%)", '180-Day Window': f"{w_comp['eligible_customers']['180d']:,} (59.9%)"},
    {'Metric': 'Total Positive Return Cases', '120-Day Window': f"{w_comp['positive_cases']['120d']:,}", '180-Day Window': f"{w_comp['positive_cases']['180d']:,}"},
    {'Metric': 'Positive Return Incidence (%)', '120-Day Window': f"{w_comp['positive_incidence_pct']['120d']:.4f}%", '180-Day Window': f"{w_comp['positive_incidence_pct']['180d']:.4f}%"},
    {'Metric': 'Positive-to-Negative Ratio', '120-Day Window': f"{w_comp['positive_to_negative_ratio']['120d']:.6f} (~0.0079)", '180-Day Window': f"{w_comp['positive_to_negative_ratio']['180d']:.6f} (~0.0119)"},
    {'Metric': 'LR PR-AUC', '120-Day Window': f"{w_comp['logistic_regression_metrics']['120d']['pr_auc']:.4f}", '180-Day Window': f"{w_comp['logistic_regression_metrics']['180d']['pr_auc']:.4f}"},
    {'Metric': 'LR ROC-AUC', '120-Day Window': f"{w_comp['logistic_regression_metrics']['120d']['roc_auc']:.4f}", '180-Day Window': f"{w_comp['logistic_regression_metrics']['180d']['roc_auc']:.4f}"},
    {'Metric': 'LR Top-Decile Lift', '120-Day Window': f"{w_comp['logistic_regression_metrics']['120d']['top_decile_lift']:.4f}x", '180-Day Window': f"{w_comp['logistic_regression_metrics']['180d']['top_decile_lift']:.4f}x"},
    {'Metric': 'LR F1-Score', '120-Day Window': f"{w_comp['logistic_regression_metrics']['120d']['f1_score']:.4f}", '180-Day Window': f"{w_comp['logistic_regression_metrics']['180d']['f1_score']:.4f}"},
    {'Metric': 'LR Train-Test PR Gap', '120-Day Window': f"{w_comp['logistic_regression_metrics']['120d']['train_test_pr_gap']:.4f}", '180-Day Window': f"{w_comp['logistic_regression_metrics']['180d']['train_test_pr_gap']:.4f}"},
    {'Metric': 'RF PR-AUC', '120-Day Window': f"{w_comp['random_forest_metrics']['120d']['pr_auc']:.4f}", '180-Day Window': f"{w_comp['random_forest_metrics']['180d']['pr_auc']:.4f}"},
    {'Metric': 'RF ROC-AUC', '120-Day Window': f"{w_comp['random_forest_metrics']['120d']['roc_auc']:.4f}", '180-Day Window': f"{w_comp['random_forest_metrics']['180d']['roc_auc']:.4f}"},
    {'Metric': 'RF Top-Decile Lift', '120-Day Window': f"{w_comp['random_forest_metrics']['120d']['top_decile_lift']:.4f}x", '180-Day Window': f"{w_comp['random_forest_metrics']['180d']['top_decile_lift']:.4f}x"},
    {'Metric': 'RF F1-Score', '120-Day Window': f"{w_comp['random_forest_metrics']['120d']['f1_score']:.4f}", '180-Day Window': f"{w_comp['random_forest_metrics']['180d']['f1_score']:.4f}"},
    {'Metric': 'RF Train-Test PR Gap', '120-Day Window': f"{w_comp['random_forest_metrics']['120d']['train_test_pr_gap']:.4f}", '180-Day Window': f"{w_comp['random_forest_metrics']['180d']['train_test_pr_gap']:.4f}"},
]

df_w_comp = pd.DataFrame(w_rows)
print(f"Window Assessment:\n{w_comp['window_assessment']}\n")
df_w_comp

## 5. Post-Evaluation Operational Assessment & Fallback Status

Assessment of candidate model operational viability against baselines and confirmation of the fallback activation.

In [ ]:
dec = cp7['methodological_decisions']

dec_summary = [
    {'Evaluation Scope': 'Predictive Churn Model Status', 'Decision Outcome': dec['churn_model_status'], 'Methodological Rationale': dec['rationale']},
    {'Evaluation Scope': 'Fallback Execution Status', 'Decision Outcome': dec['fallback_status'], 'Methodological Rationale': dec['fallback_rationale']},
    {'Evaluation Scope': 'Project Pipeline Status', 'Decision Outcome': dec['project_fallback_status'], 'Methodological Rationale': 'Retention analysis fallback successfully activated and executed; pipeline continues via empirical retention dynamics.'}
]

df_dec = pd.DataFrame(dec_summary)
pd.set_option('display.max_colwidth', None)
df_dec

## 6. Executed Fallback Results: Customer Retention Analysis

Downstream execution of the pre-registered fallback (`customer_retention_analysis`) on the 180-day window ($N=55,907$ eligible customers).
Analyzes behavioral segment re-engagement rates, spend distribution, recency-stratified retention decay, and frequency dynamics.

In [ ]:
fb = cp7.get('customer_retention_analysis_fallback_results', {})

print(f"Fallback Execution Status: {fb.get('fallback_execution_status')}")
print(f"Evaluation Window: {fb.get('evaluation_window_days')} days")
print(f"Eligible Customers Analyzed: {fb.get('total_eligible_customers_analyzed'):,}")
print(f"Total Returners Observed: {fb.get('total_returners_observed'):,} ({fb.get('overall_return_rate_pct'):.4f}%)")
print(f"Total Post-Cutoff Re-engagement Spend: R$ {fb.get('total_post_cutoff_reengagement_spend_brl'):,.2f}\n")

# 1. Behavioral Segment Re-engagement
print("--- 1. Behavioral Segment Re-engagement Dynamics ---")
seg_rows = []
for seg_name, seg_data in fb.get('behavioral_segment_reengagement', {}).items():
    seg_rows.append({
        'Segment': seg_name,
        'Eligible Customers': f"{seg_data['eligible_customers']:,}",
        'Returners': f"{seg_data['returning_customers']:,}",
        'Return Rate (%)': f"{seg_data['return_rate_pct']:.2f}%",
        'Observed Non-Return (%)': f"{seg_data['observed_non_return_rate_pct']:.2f}%",
        'Re-engagement Spend (BRL)': f"R$ {seg_data['post_cutoff_reengagement_spend_brl']:,.2f}",
        'Spend Share (%)': f"{seg_data['reengagement_spend_share_pct']:.2f}%"
    })
df_seg = pd.DataFrame(seg_rows)
display(df_seg)

# 2. Recency-Stratified Return Decay
print("\n--- 2. Recency-Stratified Return Decay ---")
rec_rows = []
for bucket, r_data in fb.get('recency_stratified_return_decay', {}).items():
    rec_rows.append({
        'Recency Interval': bucket,
        'Customer Count': f"{r_data['customer_count']:,}",
        'Returners': f"{r_data['return_count']:,}",
        'Return Rate (%)': f"{r_data['return_rate_pct']:.2f}%",
        'Observed Non-Return (%)': f"{r_data['observed_non_return_rate_pct']:.2f}%"
    })
df_rec = pd.DataFrame(rec_rows)
display(df_rec)

# 3. Frequency Tier Repurchase Dynamics
print("\n--- 3. Frequency Tier Repurchase Dynamics ---")
freq_rows = []
for tier, f_data in fb.get('frequency_tier_repurchase_dynamics', {}).items():
    freq_rows.append({
        'Frequency Tier': tier,
        'Customer Count': f"{f_data['customer_count']:,}",
        'Returners': f"{f_data['return_count']:,}",
        'Return Rate (%)': f"{f_data['return_rate_pct']:.2f}%"
    })
df_freq = pd.DataFrame(freq_rows)
display(df_freq)

## 7. Phase 7 Summary & Decision Context

### A. Pre-Registered Feasibility & Validation Design
- **Temporal Cutoffs**: Strictly leak-free design with $W=120\text{d}$ ($T_{obs}=\text{2018-05-01 15:00:37}$) and $W=180\text{d}$ ($T_{obs}=\text{2018-03-02 15:00:37}$) ensuring complete follow-up without right-censoring.
- **Protocol**: Stratified 80/20 train/test split (`random_state=42`), 5-fold cross-validation threshold tuning on training folds only, test set evaluated strictly once without synthetic oversampling (no SMOTE).
- **Baselines**: Majority-class, stratified dummy, and frozen RFM heuristic (`recency_days <= 90.0`).

### B. Actual Model Performance Evidence
- **120-Day Window**:
  - *Logistic Regression*: PR-AUC 0.0233, ROC-AUC 0.6099, Top-Decile Lift 2.4067x, Precision 0.0566, Recall 0.0556, F1 0.0561, Brier 0.22781, Train-Test PR-AUC Divergence 0.0028.
  - *Random Forest*: PR-AUC 0.0165, ROC-AUC 0.5734, Top-Decile Lift 1.8513x, Precision 0.0857, Recall 0.0278, F1 0.0420, Brier 0.16386, Train-Test PR-AUC Divergence 0.1210.
- **180-Day Window**:
  - *Logistic Regression*: PR-AUC 0.0263, ROC-AUC 0.5792, Top-Decile Lift 2.2122x, Precision 0.0727, Recall 0.0611, F1 0.0664, Brier 0.22960, Train-Test PR-AUC Divergence 0.0074.
  - *Random Forest*: PR-AUC 0.0287, ROC-AUC 0.5850, Top-Decile Lift 1.3731x, Precision 0.0645, Recall 0.0153, F1 0.0247, Brier 0.16701, Train-Test PR-AUC Divergence 0.1258.
- *Empirical Note*: Constant baselines produce nominal non-1.0x top-decile lifts solely due to arbitrary dataframe index ordering when sorting constant probability predictions; their theoretical lift is 1.00x.

### C. Post-Evaluation Operational Assessment
**Decision: POST-EVALUATION ASSESSMENT: CHURN MODEL NOT SUPPORTED FOR STANDALONE OPERATIONAL USE**

Candidate models were evaluated on held-out test data under the predefined temporal and validation design. The results show limited discrimination above the rare-event baseline, with ROC-AUC values close to 0.5 and low thresholded precision/recall. Logistic Regression showed relatively small train-test PR-AUC divergence, while tree-based models showed substantially larger divergence. No numerical operational-performance acceptance threshold was preregistered; therefore, the decision not to support standalone operational deployment is an evidence-based post-evaluation assessment rather than a preregistered predictive-performance gate.

### D. Executed Fallback
The fallback `customer_retention_analysis` was activated and successfully executed. Rather than relying on unreliable individual binary return scores, the project pivots to empirical retention curves, segment-level re-engagement economics, and lifecycle transition probabilities.

### Next Steps
- Validation complete. All checkpoints locked. Ready for Phase 8 customer retention and business recommendation reporting upon authorization.